# Pick-time calibration

Tune the pick-time regression by **physical anchors (seconds)**, not raw coefficients.
The live curves use the *real* `Pick._pick_time`, so what you tune equals what the sim charges.

Formula: `pick_time = pick_intercept + qty*(pick_weight_coef*ln(w) + pick_volume_coef*ln(v)) + cart_swap_coef*swapped`  
Travel: `|dx|*x_speed + |dy|*y_speed`.

Drag the sliders until a typical pick is ~10x a single travel step and the per-quantity shape looks right; the derived coefficients print at the bottom, ready to paste into `REGRESSION_CONFIGS` / `PickConfig`.

In [1]:
import os, sys, math, pathlib
import numpy as np
import matplotlib.pyplot as plt

# find repo root (dir containing Warehouse/) walking up from cwd
p = pathlib.Path.cwd()
while p != p.parent and not (p / 'Warehouse').exists():
    p = p.parent
ROOT = str(p)
for sub in ('Warehouse', 'Optimization'):
    d = os.path.join(ROOT, sub)
    if d not in sys.path:
        sys.path.insert(0, d)

from Pick import PickConfig, _pick_time, DEFAULT_HEIGHT_BRACKETS   # the REAL sim formula
try:
    import ipywidgets as W
    _HAVE_WIDGETS = True
except Exception:
    _HAVE_WIDGETS = False
print('repo root:', ROOT, '| widgets:', _HAVE_WIDGETS)

# Objective: minimise the expected labor of a randomly-assigned task.  Target the
# regression so the reference task lands near this labor(handling) / travel split.
TARGET_LABOR_SHARE = 0.85        # handling : travel = 85 : 15

repo root: C:\Users\myfir\Code and Repos\Inventory_Location_Optimizer | widgets: True


In [2]:
# Reference archetypes (weight, volume) spanning the carton distribution, and the
# representative aisle TASK SHAPE that sets the handling/travel split.
ITEMS = {
    'light':   (3,    27),
    'typical': (20,   27_000),
    'heavy':   (55,   110_000),
    'bulky':   (15,   110_000),
}
W0, V0 = ITEMS['typical']        # anchor item for the per-unit handling anchor
COL_STEP = 48                    # pallet column physical step (Aisle_Dimensions)
LVL_STEP = 48                    # extra_large level step

# ── task shape — the knobs that decide the handling/travel split ───────────────
# handling = PICKS · (intercept + qty·handle)   ← placement-INVARIANT (the fixed cost)
# travel   = COLUMNS · walk_col + LEVELS · walk_level   ← the ONLY lever placement pulls
# These MUST match the real workload or the split below is fiction.  The sim's real tasks
# are far more handling-heavy than a toy 8-pick task (observed ~96% handling): many picks
# per aisle, little travel.  Use actual_travel_share(<run_dir>) below to read the realised
# split from a completed run, then set REF_TASK_PICKS / COLUMNS to reproduce it before
# solving for a new target.
REF_TASK_COLUMNS = 8             # distinct columns the picker sweeps in a task
REF_TASK_LEVELS  = 4             # vertical level movements in a task
REF_TASK_PICKS   = 8             # pick stops in the task
REF_TASK_QTY     = 2             # units per pick
LEVEL_WALK_RATIO = 0.6           # walk_sec_per_level / walk_sec_per_column (solver keeps this)

In [3]:
def derive_coeffs(setup_sec, handle_sec_per_unit, weight_share,
                  walk_sec_per_column, walk_sec_per_level, cart_swap_sec):
    """Map physical anchors (seconds) -> raw PickConfig coefficients."""
    lnW, lnV = math.log(max(W0, 1)), math.log(max(V0, 1))
    pw = weight_share * handle_sec_per_unit / lnW if lnW else 0.0
    pv = (1 - weight_share) * handle_sec_per_unit / lnV if lnV else 0.0
    return dict(
        pick_intercept=setup_sec,
        pick_weight_coef=pw,
        pick_volume_coef=pv,
        x_speed=walk_sec_per_column / COL_STEP,
        y_speed=walk_sec_per_level / LVL_STEP,
        cart_swap_coef=cart_swap_sec,
    )

In [4]:
def _fmt_brackets(bs):
    """Repr the height brackets with float('inf') (not bare 'inf') so the printed
    config block is valid Python to paste into REGRESSION_CONFIGS."""
    parts = ["(float('inf'), {!r})".format(m) if math.isinf(t) else "({!r}, {!r})".format(t, m)
             for t, m in bs]
    return "(" + ", ".join(parts) + ")"


def render(setup_sec=3.0, handle_sec_per_unit=5.0, weight_share=0.7,
           walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30.0,
           config_name='calibrated',
           pick_intercept=None, pick_weight_coef=None, pick_volume_coef=None,
           x_speed=None, y_speed=None, cart_swap_coef=None):
    c = derive_coeffs(setup_sec, handle_sec_per_unit, weight_share,
                      walk_sec_per_column, walk_sec_per_level, cart_swap_sec)
    overrides = dict(pick_intercept=pick_intercept, pick_weight_coef=pick_weight_coef,
                     pick_volume_coef=pick_volume_coef, x_speed=x_speed,
                     y_speed=y_speed, cart_swap_coef=cart_swap_coef)
    for k, v in overrides.items():
        if v is not None:
            c[k] = v
    cfg = PickConfig(**c)

    qs = np.arange(1, 21)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    ax = axes[0]
    for name, (w, v) in ITEMS.items():
        ax.plot(qs, [_pick_time(cfg, w, v, int(q), False) for q in qs], marker='o', ms=3, label=name)
    ax.set_xlabel('quantity'); ax.set_ylabel('pick time (s)')
    ax.set_title('Pick time vs quantity (real _pick_time)'); ax.legend(); ax.grid(alpha=.3)

    ax = axes[1]
    ws = np.linspace(1, 60, 80)
    ax.plot(ws, [c['pick_weight_coef'] * math.log(max(w, 1)) for w in ws], label='weight term / unit')
    vs = np.linspace(27, 110_000, 80)
    ax.plot(ws, [c['pick_volume_coef'] * math.log(max(v, 1)) for v in vs], label='volume term / unit (vol swept)')
    ax.set_xlabel('weight (volume co-swept)'); ax.set_ylabel('s / unit')
    ax.set_title('Per-unit handling components'); ax.legend(fontsize=8); ax.grid(alpha=.3)

    ax = axes[2]
    w, v = ITEMS['typical']
    handling = REF_TASK_PICKS * _pick_time(cfg, w, v, REF_TASK_QTY, False)
    travel = REF_TASK_COLUMNS * COL_STEP * cfg.x_speed + REF_TASK_LEVELS * LVL_STEP * cfg.y_speed
    ax.bar(['travel', 'handling'], [travel, handling], color=['#4c72b0', '#dd8452'])
    ax.set_ylabel('time (s)'); ax.set_title('Reference task (~8 picks)'); ax.grid(axis='y', alpha=.3)
    # target labor-share line on the handling bar's expected height
    tgt_handling = TARGET_LABOR_SHARE / (1 - TARGET_LABOR_SHARE) * travel
    ax.axhline(tgt_handling, color='#c44e52', ls='--', lw=1,
               label=f'handling @ {TARGET_LABOR_SHARE:.0%} target')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    one_pick = _pick_time(cfg, w, v, REF_TASK_QTY, False)
    one_step = COL_STEP * cfg.x_speed
    tot = travel + handling
    ratio = one_pick / one_step if one_step else float('inf')
    labor_share = handling / tot if tot else float('nan')
    print(f'typical pick (qty={REF_TASK_QTY}) = {one_pick:.1f}s | one column step = {one_step:.2f}s | pick:step = {ratio:.1f}x')
    print(f'reference task: handling {handling:.0f}s ({labor_share * 100:.0f}%) vs travel {travel:.0f}s ({(1 - labor_share) * 100:.0f}%)')
    print(f'  -> target labor share {TARGET_LABOR_SHARE:.0%}; current {labor_share:.0%} '
          f'({"OK" if abs(labor_share - TARGET_LABOR_SHARE) <= 0.02 else "tune handle/walk anchors"})')
    print()
    print('# --- paste this entry into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---')
    print('    {')
    print(f"        'name'            : {config_name!r},")
    print(f"        'pick_intercept'  : {c['pick_intercept']:.6g},")
    print(f"        'pick_weight_coef': {c['pick_weight_coef']:.6g},")
    print(f"        'pick_volume_coef': {c['pick_volume_coef']:.6g},")
    print(f"        'cart_swap_coef'  : {c['cart_swap_coef']:.6g},")
    print(f"        'x_speed'         : {c['x_speed']:.6g},")
    print(f"        'y_speed'         : {c['y_speed']:.6g},")
    print(f"        'height_brackets' : {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)},")
    print('    },')
    return c

In [5]:
# Interactive: anchors drive (sliders); derived raw coefficients print below the plots.
if _HAVE_WIDGETS:
    W.interact(
        render,
        setup_sec=W.FloatSlider(value=3.0, min=0, max=30, step=0.5, description='setup s/stop'),
        handle_sec_per_unit=W.FloatSlider(value=5.0, min=0, max=60, step=0.5, description='handle s/unit'),
        weight_share=W.FloatSlider(value=0.7, min=0, max=1, step=0.05, description='weight share'),
        walk_sec_per_column=W.FloatSlider(value=0.4, min=0.0, max=5, step=0.1, description='walk s/col'),
        walk_sec_per_level=W.FloatSlider(value=0.25, min=0.0, max=5, step=0.1, description='walk s/level'),
        cart_swap_sec=W.FloatSlider(value=30.0, min=0, max=60*10, step=20, description='cart swap s'),
        config_name=W.Text(value='calibrated', description='config name'),
        pick_intercept=W.fixed(None), pick_weight_coef=W.fixed(None), pick_volume_coef=W.fixed(None),
        x_speed=W.fixed(None), y_speed=W.fixed(None), cart_swap_coef=W.fixed(None),
    )
else:
    render()

interactive(children=(FloatSlider(value=3.0, description='setup s/stop', max=30.0, step=0.5), FloatSlider(valu…

## Travel-share tools

Handling per task is **placement-invariant** (`intercept + qty·handle`, summed over picks) — you can't slot your way out of it. **Travel is the only lever placement can pull**, so the regression has to give travel a real share or no assignment policy can separate.

- **`solve_walk_for_travel_share(target, …)`** — back-computes the walk speeds (`x_speed`/`y_speed`) needed to hit a target travel share for the reference task, then renders + prints a paste-ready config.
- **`actual_travel_share(run_dir)`** — reads the *realised* travel share from a completed run. The reference task is only honest if `REF_TASK_PICKS`/`COLUMNS`/`LEVELS`/`QTY` reproduce it, so **check a real run first, align the task-shape constants, then solve.**

In [6]:
# ── TOOL 1 — solve walk speeds for a target TRAVEL share ───────────────────────
# Handling per task is placement-invariant; travel is the lever.  Hold the handling anchors
# + task shape fixed and back-solve walk_sec_per_column/level to hit a target travel share,
# then render + print a paste-ready config.
def solve_walk_for_travel_share(target_travel_share=1 - TARGET_LABOR_SHARE,
                                setup_sec=30.0, handle_sec_per_unit=10.0,
                                weight_share=0.7, cart_swap_sec=30.0,
                                config_name='calibrated'):
    s = target_travel_share
    handling = REF_TASK_PICKS * (setup_sec + REF_TASK_QTY * handle_sec_per_unit)
    travel_needed = s / (1 - s) * handling
    walk_col   = travel_needed / (REF_TASK_COLUMNS + LEVEL_WALK_RATIO * REF_TASK_LEVELS)
    walk_level = LEVEL_WALK_RATIO * walk_col
    print(f'target travel share {s:.0%}  ->  walk_sec_per_column={walk_col:.3f}  '
          f'walk_sec_per_level={walk_level:.3f}')
    print(f'(reference task: handling={handling:.0f}s, travel target={travel_needed:.0f}s)\n')
    return render(setup_sec=setup_sec, handle_sec_per_unit=handle_sec_per_unit,
                  weight_share=weight_share, walk_sec_per_column=walk_col,
                  walk_sec_per_level=walk_level, cart_swap_sec=cart_swap_sec,
                  config_name=config_name)

# ── TOOL 2 — read the ACTUAL travel share from a completed run ─────────────────
# Aligns the reference task to reality: reads per-strategy travel_time/handling_time from a
# run's stats_summary.csv (needs run_analysis to have produced stats/).  Set REF_TASK_PICKS /
# COLUMNS so render's split matches this before trusting solve_walk_for_travel_share.
def actual_travel_share(run_dir, config='calibrated'):
    import glob, csv
    tt, ht = [], []
    for f in glob.glob(os.path.join(run_dir, '*', config, 'stats', 'stats_summary.csv')):
        with open(f, encoding='utf-8') as fh:
            for r in csv.DictReader(fh):
                if r['metric'] == 'travel_time':   tt.append(float(r['mean']))
                if r['metric'] == 'handling_time': ht.append(float(r['mean']))
    if not tt:
        print('no stats_summary.csv under', run_dir, '(run run_analysis with stats first)')
        return None
    T, H = sum(tt) / len(tt), sum(ht) / len(ht)
    share = T / (T + H)
    print(f'{config}: actual travel share = {share:.1%}   '
          f'(travel {T:,.0f} / handling {H:,.0f}, n={len(tt)})')
    return share

# Example: dial the regression to 15% travel for the reference task (edit & run)
_ = solve_walk_for_travel_share(target_travel_share=0.15, setup_sec=30.0, handle_sec_per_unit=10.0)
# Example: check what a real run actually produced (edit the path, then set REF_TASK_* to match):
# actual_travel_share(r'H:\Data\Inventory_Optimizer_Data\Optimization_Outputs\comparison_20260619_172724')

target travel share 15%  ->  walk_sec_per_column=6.787  walk_sec_per_level=4.072
(reference task: handling=400s, travel target=71s)



typical pick (qty=2) = 50.0s | one column step = 6.79s | pick:step = 7.4x
reference task: handling 400s (85%) vs travel 71s (15%)
  -> target labor share 85%; current 85% (OK)

# --- paste this entry into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---
    {
        'name'            : 'calibrated',
        'pick_intercept'  : 30,
        'pick_weight_coef': 2.33666,
        'pick_volume_coef': 0.294014,
        'cart_swap_coef'  : 30,
        'x_speed'         : 0.141403,
        'y_speed'         : 0.0848416,
        'height_brackets' : ((96.0, 1.0), (240.0, 1.2), (float('inf'), 1.4)),
    },


C:\Users\myfir\AppData\Local\Temp\ipykernel_25340\3698779675.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Override a raw coefficient directly
The sliders are physical anchors; to hand-tune a raw coefficient, call `render` with it explicitly (overrides the derived value), e.g.:
```python
render(setup_sec=3, handle_sec_per_unit=5, weight_share=0.7,
       walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30,
       pick_weight_coef=1.6)   # <- raw override
```

In [7]:
# scratch: direct raw-coefficient override example (edit + run)
_ = render(setup_sec=30.0, handle_sec_per_unit=10.0, weight_share=0.7,
           walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30.0)

typical pick (qty=2) = 50.0s | one column step = 0.40s | pick:step = 125.0x
reference task: handling 400s (99%) vs travel 4s (1%)
  -> target labor share 85%; current 99% (tune handle/walk anchors)

# --- paste this entry into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---
    {
        'name'            : 'calibrated',
        'pick_intercept'  : 30,
        'pick_weight_coef': 2.33666,
        'pick_volume_coef': 0.294014,
        'cart_swap_coef'  : 30,
        'x_speed'         : 0.00833333,
        'y_speed'         : 0.00520833,
        'height_brackets' : ((96.0, 1.0), (240.0, 1.2), (float('inf'), 1.4)),
    },


C:\Users\myfir\AppData\Local\Temp\ipykernel_25340\3698779675.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
